# 4. Analysis

$$
\def\thot{{{\color{red}\theta_{13}}}}
\def\thtt{{{\color{blue}\theta_{23}}}}
\def\dmto{{{\color{orange}\Delta{}m^2_{31}}}}
\def\dmtwo{{{\color{cyan}\Delta{}m^2_{21}}}}
\def\dmsol{{{\color{cyan}\Delta{}m^2_\mathrm{sol}}}}
\def\dmtt{{{\color{purple}\Delta{}m^2_{32}}}}
\def\dmatm{{{\color{purple}\Delta{}m^2_\mathrm{atm}}}}
\def\dcp{{{\color{green}\delta_\mathrm{CP}}}}
\require{ams}
$$
Now we will put all we've learned together to try and measure $\dcp$ from some observations.

The only additional bit of information you might need is that your test statistic, $T$, over multiple samples is just the sum of the test statistic over each, like:

Below we provide the model `PredictSamples`, you might notice that the argument list is slightly different. I noticed that the original function required you to recalculate the reconstructed variables each time you changed the oscillation parameters, which lead to an unsmooth likelihood that was hard to fit.

```python
Tall = Pearson_N2LLH(obs_hist1, pred_hist1) + Pearson_N2LLH(obs_hist2, pred_hist2) + Pearson_N2LLH( ...
```
<div style="display:None;">
<span style="color:#800080;font-weight:bold;">Below you'll need to copy your `PredictSamples` function from [C Analysing Neutrino Events](./3EventSamples.ipynb), which should return four predicted histograms. You might need to update the binning to match that used by the observations. You will need to includethe new `pred_to_obsscale` parameter, which should be added to the histogram weights as below:</span>

```python
   numode_numu_cc_hist = hist1d(data=reco_E_nu_numu, weights=event_osc_weights_numu_surv*pred_to_obsscale, bins=bins)
```

This scale factor is just to account for the fact that we usually simulate many more events for the prediction than we record in the observation. <span style="color:#6699cc;font-weight:bold;">Why do you think this might be?</span></div>

In [ ]:
import pandas as pa
from proj_utils import *
import matplotlib.pyplot as plt

obs_bins = ana.get_bins()
nofobs_nu_mode_numucc = ana.get_nofobs_nu_mode_numucc()
nofobs_nu_mode_nuecc = ana.get_nofobs_nu_mode_nuecc()
nofobs_antinu_mode_numucc = ana.get_nofobs_antinu_mode_numucc()
nofobs_antinu_mode_nuecc = ana.get_nofobs_antinu_mode_nuecc()

obs_nu_mode_numucc = ana.get_obs_nu_mode_numucc()
obs_nu_mode_nuecc = ana.get_obs_nu_mode_nuecc()
obs_antinu_mode_numucc = ana.get_obs_antinu_mode_numucc()
obs_antinu_mode_nuecc = ana.get_obs_antinu_mode_nuecc()

simulated_events_nu_mode = pa.read_csv("simulation/neutrino_mode_events.csv")
simulated_events_antinu_mode = pa.read_csv("simulation/antineutrino_mode_events.csv")

DUNE_baseline = 1300 # km

events_nu_numu_cc = simulated_events_nu_mode[simulated_events_nu_mode["pid_lepton"] == 13]
events_nu_nue_cc = simulated_events_nu_mode[simulated_events_nu_mode["pid_lepton"] == 11]
events_nu_antinumu_cc = simulated_events_antinu_mode[simulated_events_antinu_mode["pid_lepton"] == -13]
events_nu_antinue_cc = simulated_events_antinu_mode[simulated_events_antinu_mode["pid_lepton"] == -11]

smearing_params = {  "muon_energy_resolution": 0.02, # 2%
                 "electron_energy_resolution": 0.05, #
                 "proton_kinetic_energy_resolution": 0.1, #
                 "charged_pion_kinetic_energy_resolution": 0.1, #
                 "charged_pion_mass_visible_fraction": 0.75, #
                 "neutral_pion_energy_resolution": 0.1 #
              }

reco_E_nu_numu = ReconstructedNeutrinoEnergy(events_nu_numu_cc, smearing_params)
reco_E_nu_nue = ReconstructedNeutrinoEnergy(events_nu_nue_cc, smearing_params)
reco_E_antinu_numu = ReconstructedNeutrinoEnergy(events_nu_antinumu_cc, smearing_params)
reco_E_antinu_nue = ReconstructedNeutrinoEnergy(events_nu_antinue_cc, smearing_params)

def PredictSamples(simulated_events_nu_mode, simulated_events_antinu_mode, osc_params, pred_to_obsscale=0.15):

    event_osc_weights_numu_surv = \
        Probability_Matter_LBL(events_nu_numu_cc["E_neutrino"], DUNE_baseline, 
                               osc_params, osc_channels=["numu_survival"])
    event_osc_weights_nue_app = \
        Probability_Matter_LBL(events_nu_nue_cc["E_neutrino"], DUNE_baseline, 
                               osc_params, osc_channels=["nue_appearance"])
    
    event_osc_weights_antinumu_surv = \
        Probability_Matter_LBL(events_nu_antinumu_cc["E_neutrino"], DUNE_baseline, 
                               osc_params, osc_channels=["antinumu_survival"])
    event_osc_weights_antinue_app = \
        Probability_Matter_LBL(events_nu_antinue_cc["E_neutrino"], DUNE_baseline, 
                               osc_params, osc_channels=["antinue_appearance"])
    
    return (hist1d(data=reco_E_nu_numu, weights=event_osc_weights_numu_surv*pred_to_obsscale, bins=obs_bins), \
            hist1d(data=reco_E_nu_nue, weights=event_osc_weights_nue_app*pred_to_obsscale, bins=obs_bins)), \
           (hist1d(data=reco_E_antinu_numu, weights=event_osc_weights_antinumu_surv*pred_to_obsscale, bins=obs_bins), \
            hist1d(data=reco_E_antinu_nue, weights=event_osc_weights_antinue_app*pred_to_obsscale, bins=obs_bins)),

In [ ]:
osc_params = {
  "s12sq": 0.31,
  "s13sq": 0.02,
  "s23sq": 0.55,
  "delta": 0.7 * np.pi,
  "Dmsq21": 7.5e-5,
  "Dmsq31": 2.5e-3
}

(pred_nu_mode_numucc, pred_nu_mode_nuecc), (pred_antinu_mode_numucc, pred_antinu_mode_nuecc) = \
    PredictSamples(simulated_events_nu_mode, simulated_events_antinu_mode, osc_params)

drawhist1d(hist=obs_nu_mode_numucc, label="Observation")
drawhist1d(hist=nofobs_nu_mode_numucc, label="Observation (No Poiss Fluct)")
drawhist1d(hist=pred_nu_mode_numucc, label="Prediction")
plt.legend()
plt.show()

drawhist1d(hist=obs_nu_mode_nuecc, label="Observation")
drawhist1d(hist=nofobs_nu_mode_nuecc, label="Observation (No Poiss Fluct)")
drawhist1d(hist=pred_nu_mode_nuecc, label="Prediction")
plt.legend()
plt.show()

drawhist1d(hist=obs_antinu_mode_numucc, label="Observation")
drawhist1d(hist=nofobs_antinu_mode_numucc, label="Observation (No Poiss Fluct)")
drawhist1d(hist=pred_antinu_mode_numucc, label="Prediction")
plt.legend()
plt.show()

drawhist1d(hist=obs_antinu_mode_nuecc, label="Observation")
drawhist1d(hist=nofobs_antinu_mode_nuecc, label="Observation (No Poiss Fluct)")
drawhist1d(hist=pred_antinu_mode_nuecc, label="Prediction")
plt.legend()
plt.show()

# 4.1 Measure The CP-violating phase

Can you exclude CP conservation ($\sin\delta_\mathrm{CP} = -1, 0, 1$) for the above data? At what confidence level?

<div style="display:None;">
    def lhood(osc_params):
    smearing_params = {  "muon_energy_resolution": 0.02, # 2%
                 "electron_energy_resolution": 0.05, #
                 "proton_kinetic_energy_resolution": 0.1, #
                 "charged_pion_kinetic_energy_resolution": 0.1, #
                 "charged_pion_mass_visible_fraction": 0.75, #
                 "neutral_pion_energy_resolution": 0.1 #
              }

    (pred_nu_mode_numucc, pred_nu_mode_nuecc), (pred_antinu_mode_numucc, pred_antinu_mode_nuecc) = \
        PredictSamples(simulated_events_nu_mode, simulated_events_antinu_mode, osc_params)

    return np.array([ Pearson_N2LLH(obs_nu_mode_numucc[0],pred_nu_mode_numucc[0]), \
                      Pearson_N2LLH(obs_nu_mode_nuecc[0],pred_nu_mode_nuecc[0]), \
                      Pearson_N2LLH(obs_antinu_mode_numucc[0],pred_antinu_mode_numucc[0]), \
                      Pearson_N2LLH(obs_antinu_mode_nuecc[0],pred_antinu_mode_nuecc[0]) ] )

osc_params = {
  "s12sq": 0.31,
  "s13sq": 0.02,
  "s23sq": 0.55,
  "delta": 0.7 * np.pi,
  "Dmsq21": 7.5e-5,
  "Dmsq31": 2.5e-3
}

vop = osc_params.copy()

fit_params = ["s13sq", "s23sq", "delta", "Dmsq31"]
param_ranges = [ [0.0125, 0.0275], [0.5,0.6] , [-1*np.pi ,1*np.pi], [2.4E-3, 2.6E-3]]

for i, fp in enumerate(fit_params):
    vop = osc_params.copy()
    nsteps = 100
    lhoods = np.zeros(nsteps)
    for j, v in enumerate(np.linspace(param_ranges[i][0], param_ranges[i][1], nsteps)):
        vop[fp] = v
        lhoods[j] = np.sum(lhood(vop))
    plt.plot(np.linspace(param_ranges[i][0], param_ranges[i][1], nsteps), lhoods, label=fp)
    plt.legend()
    plt.show()
</div>

In [ ]:
def lhood(osc_params, fluct=True):
    (pred_nu_mode_numucc, pred_nu_mode_nuecc), (pred_antinu_mode_numucc, pred_antinu_mode_nuecc) = \
        PredictSamples(simulated_events_nu_mode, simulated_events_antinu_mode, osc_params)
    if fluct:
        return np.array([ Pearson_N2LLH(obs_nu_mode_numucc[0],pred_nu_mode_numucc[0]), \
                          Pearson_N2LLH(obs_nu_mode_nuecc[0],pred_nu_mode_nuecc[0]), \
                          Pearson_N2LLH(obs_antinu_mode_numucc[0],pred_antinu_mode_numucc[0]), \
                          Pearson_N2LLH(obs_antinu_mode_nuecc[0],pred_antinu_mode_nuecc[0]) ] )
    else:
        return np.array([ Pearson_N2LLH(nofobs_nu_mode_numucc[0],pred_nu_mode_numucc[0]), \
                          Pearson_N2LLH(nofobs_nu_mode_nuecc[0],pred_nu_mode_nuecc[0]), \
                          Pearson_N2LLH(nofobs_antinu_mode_numucc[0],pred_antinu_mode_numucc[0]), \
                          Pearson_N2LLH(nofobs_antinu_mode_nuecc[0],pred_antinu_mode_nuecc[0]) ] )

In [ ]:
osc_params = {
  "s12sq": 0.31,
  "s13sq": 0.02,
  "s23sq": 0.55,
  "delta": 0.7 * np.pi,
  "Dmsq21": 7.5e-5,
  "Dmsq31": 2.5e-3
}

vop = osc_params.copy()

fit_params = ["s13sq", "s23sq", "delta", "Dmsq31"]
param_ranges = [ [0.0125, 0.0275], [0.5, 0.6] , [-1*np.pi ,1*np.pi], [2.4E-3, 2.6E-3] ]

for i, fp in enumerate(fit_params):
    vop = osc_params.copy()
    nsteps = 100
    lhoods = np.zeros(nsteps)
    lhoods_nof = np.zeros(nsteps)
    for j, v in enumerate(np.linspace(param_ranges[i][0], param_ranges[i][1], nsteps)):
        vop[fp] = v
        lhoods[j] = np.sum(lhood(vop))
        lhoods_nof[j] = np.sum(lhood(vop,False))
    plt.plot(np.linspace(param_ranges[i][0], param_ranges[i][1], nsteps), lhoods, label=fp)
    plt.plot(np.linspace(param_ranges[i][0], param_ranges[i][1], nsteps), lhoods_nof, label=f"{fp} - No Poiss. Fluct.")
    plt.legend()
    plt.show()

In [ ]:
%%time
import scipy.optimize as op

def fng(osc_params, fluct=True):
    vop = osc_params.copy()
    def fn(x):
        nonlocal vop
        vop["s13sq"] = x[0]
        vop["s23sq"] = x[1]
        vop["Dmsq31"] = x[2]
        return np.sum(lhood(vop, fluct))
    return fn

def mini(osc_params, fluct=True):
    return op.minimize(fng(osc_params, fluct), 
                      [ vop["s13sq"], vop["s23sq"], vop["Dmsq31"] ], 
                      method='Nelder-Mead',
                      bounds=[(0.0125, 0.0275),(0.5,0.6),(2.4E-3,2.6E-3)] )

deltas = np.linspace(-1,1,100)
lhood_noprofile = np.zeros_like(deltas)
lhood_profile = np.zeros_like(deltas)
lhood_profile_nof = np.zeros_like(deltas)
for i,d in enumerate(deltas):
    vop = osc_params.copy()
    vop["delta"] = d*np.pi
    lhood_noprofile[i] = np.sum(lhood(vop))
    res = mini(vop)
    if not res.success:
        raise RuntimeError(f"{d}\n{res}")
    resnof = mini(vop, False)
    lhood_profile[i] = res.fun
    lhood_profile_nof[i] = resnof.fun

In [ ]:
true_params = {
  "s12sq": 0.31,
  "s13sq": 0.018,
  "s23sq": 0.535,
  "delta": 0.15 * np.pi,
  "Dmsq21": 7.5e-5,
  "Dmsq31": 2.435e-3
}

In [ ]:
plt.plot(deltas*np.pi, lhood_noprofile - np.min(lhood_noprofile), label="no profile")

bf = deltas[np.argmin(lhood_profile)]
plt.plot(deltas, lhood_profile - np.min(lhood_profile), label=f"profile: bf @ {bf:.2f} $\\pi$")
bf = deltas[np.argmin(lhood_profile_nof)]
plt.plot(deltas, lhood_profile_nof - np.min(lhood_profile_nof), label=f"profile (No Poiss Fluct): bf @ {bf:.2f} $\\pi$")
plt.ylim([0,10])
plt.legend()
plt.tick_params(top=True, labeltop=True)
plt.grid()
plt.plot([-np.pi,np.pi],[1,1], c="black")
plt.plot([-np.pi,np.pi],[4,4], c="black")
plt.plot([-np.pi,np.pi],[9,9], c="black")
plt.show()